# Session 3 — From Model to Clinic: Deployment and Integration

*From Pixels to Patients · AAPM 2026 (Session 3 of 3)*

A trained model can't tell you when to distrust its own output. This notebook builds the
QC layer that can — **reference-free**, using only the contour and its CT. We:

1. **Characterize** the Session 1 training cohort as a distribution.
2. **Build** a check with two orthogonal axes — **shape** (radiomic outlier?) and
   **position** (where a GTV normally sits?) — plus **input validation**.
3. **Run** it on held-out cases and watch it catch synthetic silent failures.

Method: Elguindi et al., *Reference-Free QC of Organ-at-Risk Contours via Radiomic and
Positional Signatures*. No ground truth is used at scoring time — the training
*distribution* is the reference.

In [ ]:
# Dependencies (no GPU, no trained model needed — the check reasons about the data)
%pip install -q numpy scipy nibabel pandas matplotlib

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Make the Session 3 package importable whether you launch from the repo root or here.
for base in (Path.cwd(), *Path.cwd().parents):
    if (base / 'Session3' / 'qc_workflow').is_dir():
        sys.path.insert(0, str(base / 'Session3')); break
    if (base / 'qc_workflow').is_dir():
        sys.path.insert(0, str(base)); break

from qc_workflow import (QCConfig, discover_cases, find_nifti_root, split_by_patient,
                         characterize_cohort, build_reference, load_case, check_contour)
from run_qc_checks import _corrupt_variants
cfg = QCConfig()
cfg

## 1 · Find the Session 1 export and split by patient

We reuse the exact NIfTI tree Session 2 trained on, and split **by patient** so no
patient contributes to both the reference and the held-out set (the same leakage rule).

In [ ]:
nifti_root = find_nifti_root() or cfg.nifti_root
records = discover_cases(nifti_root, required_masks=(cfg.target_mask,))
print(f'{len(records)} cases under {nifti_root}')
assert len(records) >= 4, 'Run Session 1 export first (a few NSCLC-Radiomics patients).'
splits = split_by_patient(records, holdout_fraction=0.3, seed=cfg.seed)
print('reference:', len(splits['reference']), '| holdout:', len(splits['holdout']))

## 2 · Characterize the training data

The cohort *is* the reference. We distill it into acquisition ranges (spacing, HU) and the
GTV's shape + positional distributions — the same instinct as Session 1's manifest QC, but
turned into something we can score against later.

In [ ]:
profile = characterize_cohort(splits['reference'], cfg)
print(profile.summary())

In [ ]:
# The GTV volume distribution is the simplest slice of the training profile.
vols = profile.feature_matrix[:, profile.feature_names.index('volume_cc')]
plt.figure(figsize=(5, 3))
plt.hist(vols, bins=12, color='#4C72B0', edgecolor='white')
plt.xlabel('GTV volume (cc)'); plt.ylabel('reference cases'); plt.title('Training GTV volume distribution')
plt.tight_layout(); plt.show()

## 3 · Build the reference-free check

Freeze the shape OOD model (robust Mahalanobis over radiomic features) and the position
atlas (a generalized-Procrustes constellation of GTV + lungs/heart/cord/esophagus) into one
portable object. This is what would ship next to the model.

In [ ]:
reference = build_reference(profile, cfg)
reference.save(Path(cfg.output_dir) / 'reference_check.json')
print('atlas organs:', sorted(reference.atlas.get('template', {})))
print('shape features:', len(reference.shape['feature_names']))

## 4 · Validate inputs and check outputs on held-out cases

For each held-out GTV we score **input** (is it CT? plausible spacing?), **shape**, and
**position**, and take the worst band. A normal case should pass.

In [ ]:
def context_centroids(masks, spacing):
    ctx = {}
    for m in cfg.context_masks:
        arr = masks.get(m)
        if arr is not None and arr.sum() > 0:
            zz, yy, xx = np.where(arr > 0.5)
            ctx[m] = [float(zz.mean()*spacing[0]), float(yy.mean()*spacing[1]), float(xx.mean()*spacing[2])]
    return ctx

for rec in splits['holdout']:
    image, masks, spacing = load_case(rec, (cfg.target_mask, *cfg.context_masks))
    rep = check_contour(image, masks.get(cfg.target_mask), context_centroids(masks, spacing),
                        spacing, reference, cfg, profile.spacing)
    print(f"{rec.case_id[:36]:<36} {rep['verdict'].upper():<7} " + ('; '.join(rep['reasons']) or 'looks normal'))

## 5 · Watch it catch silent failures

We take one good case and corrupt it three ways — a **mislocated** GTV, an **over-segmented**
GTV, and a **non-CT** input — and confirm each axis fires: position, shape, and input
validation respectively.

In [ ]:
demo = splits['holdout'][0]
image, masks, spacing = load_case(demo, (cfg.target_mask, *cfg.context_masks))
ctx = context_centroids(masks, spacing)
for kind, (img, msk) in _corrupt_variants(image, masks[cfg.target_mask], spacing).items():
    rep = check_contour(img, msk, ctx, spacing, reference, cfg, profile.spacing)
    print(f"{kind:<16} {rep['verdict'].upper():<7} shape z={rep['shape'].get('z')}  pos z={rep['position'].get('z')}")
    print('   ->', '; '.join(rep['reasons']))

## 6 · Extend to the whole structure set (GTV + OARs)

Assuming Session 2 also trains the thoracic OARs, we fit a **panel**: one shape reference
per structure plus a single shared positional atlas. OARs are anatomically stereotyped, so
their shape distributions are tight and outliers are meaningful.

In [ ]:
from qc_workflow import characterize_panel, build_panel, check_panel
prof = characterize_panel(splits['reference'], cfg)
panel = build_panel(prof, cfg)
panel.save(Path(cfg.output_dir) / 'panel_check.json')
for s in panel.structures:
    print(f'  {s:<10} {len(prof.feature_matrices[s]):>3} reference examples')

In [ ]:
# Score every structure on a held-out case
rec = splits['holdout'][0]
image, masks, spacing = load_case(rec, (cfg.target_mask, *cfg.context_masks))
for name, rep in check_panel(image, masks, spacing, panel, cfg, prof.spacing).items():
    print(f"  {name:<10} {rep['verdict'].upper():<7} " + ('; '.join(rep.get('reasons', [])) or 'ok'))

## 7 · Catch a swapped Lung_L / Lung_R

The classic OAR labelling error. Swapping the two lung masks should flag **both** as
wrong-side — each lands where its partner belongs.

In [ ]:
swapped = dict(masks)
swapped['lung_l'], swapped['lung_r'] = masks['lung_r'], masks['lung_l']
res = check_panel(image, swapped, spacing, panel, cfg, prof.spacing)
for n in ('lung_l', 'lung_r'):
    print(f"  {n:<8} {res[n]['verdict'].upper():<7} {'; '.join(res[n]['reasons'])}")

## 8 · Package for governance

Write the training profile, the frozen panel, a per-structure report, and a QC card —
the artifacts a monitoring/governance process consumes. (`run_qc_checks.py` does all of
this in one call.)

In [ ]:
import subprocess, sys as _sys
root = next(b for b in (Path.cwd(), *Path.cwd().parents) if (b / 'Session3').is_dir())
print(subprocess.run([_sys.executable, str(root / 'Session3' / 'run_qc_checks.py')],
                     capture_output=True, text=True).stdout)

---
### Scope
A workshop artifact, not a clinical device: a **prioritizer for human review**, trained on
**real CT only** (MR / MR-derived pseudo-CT is rejected at input validation). On a small
reference cohort it is illustrative, not reliable — scale the cohort before you trust a verdict.

**Companion package:** `contour-qc`. **Prev:** Session 2 built the model this session guards.